In [ ]:
import os
import glob
import utils
import numpy as np

print("starting script", flush=True)
base_path = "../data/rsds_past"

files_past = []
files_future = []

for root, dirs, files in os.walk(base_path):
    if not dirs:
        rel_path = os.path.relpath(root, base_path)
        path_past = os.path.join(base_path, rel_path)
        future_path = path_past.replace("rsds_past", "rsds_future")

        # Find nc files
        past_files = glob.glob(os.path.join(path_past, "*r1i1p1_1995*.nc"))
        future_files = glob.glob(os.path.join(future_path, "*r1i1p1_2045*.nc"))

        files_past.extend(past_files)
        files_future.extend(future_files)


# Use the file names as model identifiers
model_names_past = [
    f"{os.path.normpath(f).split(os.sep)[-5]}_{os.path.normpath(f).split(os.sep)[-4]}"
    for f in files_past
]
model_names_future = [
    f"{os.path.normpath(f).split(os.sep)[-5]}_{os.path.normpath(f).split(os.sep)[-4]}"
    for f in files_future
]

reference_bc = "../data/ERA/rsds/ERA/every_third_rsds_lowres_1995-2004.nc"
files_past.append(reference_bc)
files_raw_past = utils.pre_process(files_past, ["rsds"], 0, 5, 0, 5)
files_past = utils.select_time_frame(files_raw_past, slice("1995-01-01", "2004-12-30"))
model_names_past.append("ERA5")
print("File source loaded", flush=True)

In [ ]:
mape = utils.mape_average(files_past, len(files_past) - 1)
corr = utils.correlation(files_past, len(files_past) - 1)

In [ ]:
mean_arr = np.array(mape)

# Get indices of 15 smallest values
bottom15_indices = np.argpartition(mean_arr, 15)[:15]  # first 15 smallest

# Sort these indices by their values (ascending)
bottom15_indices_sorted = bottom15_indices[np.argsort(mean_arr[bottom15_indices])]

# Get the corresponding bottom 15 values
bottom15_values = mean_arr[bottom15_indices_sorted]

print("Bottom 15 indices:", bottom15_indices_sorted)
print("Bottom 15 values:", bottom15_values)

In [ ]:
selected_elements = [model_names_past[i] for i in bottom15_indices_sorted]
print(selected_elements)

In [ ]:
corr_arr = np.array(corr)
# Get indices of top 15 values
top15_indices = np.argpartition(corr_arr, -15)[-15:]

# Sort these indices by their values (descending)
top15_indices_sorted = top15_indices[np.argsort(corr_arr[top15_indices])[::-1]]

# Get the corresponding top 15 values
top15_values = corr_arr[top15_indices_sorted]

print("Top 15 indices:", top15_indices_sorted)
print("Top 15 values:", top15_values)

In [ ]:
selected_elements = [model_names_past[i] for i in top15_indices_sorted]
print(selected_elements)